# Fine-tuning IndoBERT untuk Deteksi Hoaks Bahasa Indonesia

Notebook ini dirancang untuk dijalankan di **Google Colab (GPU gratis)**. Tujuannya:

1. Memuat data dari folder `data/` (yang diunggah pengguna):
   - `train_id_oos.csv` — data latih (sumber Rifky, out-of-source train)
   - `test_id_oos.csv`  — data uji (sumber Mendeley/ICTS-2017, **tidak pernah dilatih**)
   - `holdout_id_v2.csv` — hold-out independen (103 baris, **tidak pernah dilatih/dituning**)
2. Fine-tuning `indobenchmark/indobert-base-p1` (3 epoch, GPU).
3. Evaluasi pada test out-of-source dan hold-out: akurasi, ROC-AUC, F1 per kelas,
   FPR (valid diprediksi hoak), Brier score, interval kepercayaan 95% (Wilson).
4. Menyimpan model + metrik ke `models/indobert/` untuk dimuat oleh `app.py`.

**Peringatan keandalan:** model dianggap andal HANYA jika akurasi hold-out Indonesia ≥ 85%.
Hasil TAHAP 2 (model sebelumnya) menunjukkan terbaik hanya 67,96% (< 85%),
jadi jangan mengklaim model andal sebelum metrik terpenuhi.

In [ ]:
# Instal dependensi (hanya perlu sekali per runtime Colab)
!pip -q install transformers datasets evaluate scikit-learn torch sentencepiece protobuf

## Unggah berkas data
Unggah ketiga berkas berikut dari folder `data/` lokal Anda ke Colab (tab file → upload):
- `data/train_id_oos.csv`
- `data/test_id_oos.csv`
- `data/holdout_id_v2.csv`

Atau pasang Google Drive dan tuliskan jalurnya di bawah.

In [ ]:
import os, json, math, glob
import pandas as pd
import numpy as np

# Cari berkas CSV yang diunggah (Colav: biasanya di /content)
candidates = sorted(glob.glob('/content/*_oos.csv') + glob.glob('/content/*holdout*.csv') + glob.glob('/content/*.csv'))
print('Berkas CSV di /content:')
for c in candidates:
    print(' ', c)

TRAIN_PATH = '/content/train_id_oos.csv'
TEST_PATH = '/content/test_id_oos.csv'
HOLDOUT_PATH = '/content/holdout_id_v2.csv'

for p in (TRAIN_PATH, TEST_PATH, HOLDOUT_PATH):
    assert os.path.exists(p), f'Berkas tidak ditemukan: {p}. Unggah terlebih dahulu.'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
holdout = pd.read_csv(HOLDOUT_PATH)

for df in (train, test, holdout):
    df['label'] = df['label'].astype(int)
    assert 'text' in df.columns and 'label' in df.columns

print(f"Train : {len(train)} baris | valid={(train.label==0).sum()} | hoak={(train.label==1).sum()}")
print(f"Test  : {len(test)} baris | valid={(test.label==0).sum()} | hoak={(test.label==1).sum()}")
print(f"Holdout: {len(holdout)} baris | valid={(holdout.label==0).sum()} | hoak={(holdout.label==1).sum()}")

# Pembagian sudah by-source (out-of-source): jangan ulang split di sini.
# Train = Rifky, Test = Mendeley, Hold-out = manual. Tidak ada kebocoran data.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from transformers import DataCollatorWithPadding

MODEL_NAME = 'indobenchmark/indobert-base-p1'
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'GPU tidak tersedia. Ubah runtime Colab ke GPU (Runtime → Change runtime type → GPU).'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'valid', 1: 'hoak'},
    label2id={'valid': 0, 'hoak': 1},
).to(device)

class NewsDS(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, padding=True,
                             max_length=MAX_LEN, return_tensors='pt')
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(NewsDS(train['text'], train['label']),
                           batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=collator)
test_loader = DataLoader(NewsDS(test['text'], test['label']),
                          batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collator)
holdout_loader = DataLoader(NewsDS(holdout['text'], holdout['label']),
                             batch_size=BATCH_SIZE, shuffle=False,
                             collate_fn=collator)
print('DataLoader siap. Train batches:', len(train_loader))


In [ ]:
from transformers import get_linear_schedule_with_warmup, AdamW

optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps
)

model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        loss = out.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/{EPOCHS} - avg train loss: {total_loss/len(train_loader):.4f}")

print('Fine-tuning selesai.')


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, brier_score_loss

def evaluate(loader, name):
    model.eval()
    y_true, y_prob, y_pred = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            logits = model(**batch).logits
            prob = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            pred = (prob >= 0.5).astype(int)
            y_true.extend(labels.cpu().numpy().tolist())
            y_prob.extend(prob.tolist())
            y_pred.extend(pred.tolist())
    y_true = np.array(y_true); y_prob = np.array(y_prob); y_pred = np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float('nan')
    f1_hoak = f1_score(y_true, y_pred, pos_label=1)
    f1_valid = f1_score(y_true, y_pred, pos_label=0)
    rec_hoak = f1_score(y_true, y_pred, pos_label=1) if False else (y_pred[y_true==1]==1).mean() if (y_true==1).any() else 0.0
    # FPR = P(pred=hoak | actual=valid)
    valid_mask = (y_true == 0)
    fpr = (y_pred[valid_mask] == 1).mean() if valid_mask.any() else 0.0
    brier = brier_score_loss(y_true, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    gray = None  # dihitung dengan threshold 0.30/0.70 di bawah
    print(f"\n=== {name} (n={len(y_true)}) ===")
    print(f"Akurasi : {acc:.4f}")
    print(f"ROC-AUC : {auc:.4f}")
    print(f"F1 hoak  : {f1_hoak:.4f} | F1 valid : {f1_valid:.4f}")
    print(f"Recall hoak : {rec_hoak:.4f}")
    print(f"FPR (valid→hoak) : {fpr:.4f}")
    print(f"Brier score : {brier:.4f}")
    print(f"Confusion [tn fp; fn tp] = [{tn} {fp}; {fn} {tp}]")
    return {
        'name': name, 'n': int(len(y_true)), 'accuracy': float(acc),
        'roc_auc': float(auc), 'f1_hoak': float(f1_hoak), 'f1_valid': float(f1_valid),
        'recall_hoak': float(rec_hoak), 'fpr': float(fpr), 'brier': float(brier),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

res_test = evaluate(test_loader, 'Test out-of-source (Mendeley)')
res_holdout = evaluate(holdout_loader, 'Hold-out v2 (manual, independen)')


In [ ]:
# Interval kepercayaan 95% (Wilson) untuk akurasi hold-out
def wilson_ci(p, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    denom = 1 + z*z/n
    centre = p + z*z/(2*n)
    half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return ((centre-half)/denom, (centre+half)/denom)

acc_h = res_holdout['accuracy']; n_h = res_holdout['n']
lo, hi = wilson_ci(acc_h, n_h)
print(f"Hold-out akurasi: {acc_h:.2%}")
print(f"Wilson CI 95%   : [{lo:.2%}, {hi:.2%}]")
print(f"Model andal (≥85%)? {"YA" if acc_h >= 0.85 else "TIDAK"}")


In [ ]:
# Simpan model + metrik ke models/indobert (unduh dari Colab, lalu taruh di repo)
import os

OUT_DIR = '/content/indobert_model'
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

metrics = {
    'model': MODEL_NAME,
    'epochs': EPOCHS,
    'test_out_of_source': res_test,
    'holdout_v2': res_holdout,
    'holdout_wilson_ci_95': [lo, hi],
    'reliable': bool(acc_h >= 0.85),
    'note': 'MODEL_RELIABLE di app.py ditetapkan oleh metrik di model card ini (hold-out ≥ 85%).',
}
with open(os.path.join(OUT_DIR, 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"Disimpan di {OUT_DIR}")
print("Unduh folder ini, ekstrak, lalu letakkan di models/indobert/ di repo.")
print("Berkas: config.json, model.safetensors (atau pytorch_model.bin), tokenizer.*, metrics.json")


## Langkah selanjutnya (di repo lokal)

1. Unduh folder `indobert_model` dari Colab.
2. Letakkan isinya di `models/indobert/` di repo `fake-news-detector`.
3. Pastikan `app.py` memuat model dari `models/indobert/` bila folder ada (loader opsional sudah disiapkan di TAHAP C).
4. `MODEL_RELIABLE` tetap ditentukan oleh metrik di model card (hold-out ≥ 85%), bukan otomatis hanya karena model tersedia.

Lihat `README_colab.md` untuk panduan lengkap.